# Data Exploration

This notebook explores the four benchmarks used in the DTR paper:
- **AIME 2025** - American Invitational Mathematics Examination
- **HMMT 2025** - Harvard-MIT Mathematics Tournament
- **GPQA Diamond** - Graduate-level science questions
- **LiveCodeBench** - Competitive programming problems

We examine sample counts, question length distributions, and display example questions from each benchmark.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.data.loaders import load_benchmark
from dtr.utils.io import load_json, load_jsonl

## Load All Benchmarks

We load each benchmark from the local data cache directory.

In [ ]:
BENCHMARKS = ["aime_2025", "hmmt_2025", "gpqa_diamond", "livecodebench"]
DATA_DIR = Path("..") / "data_cache"

datasets = {}
for bench in BENCHMARKS:
    datasets[bench] = load_benchmark(bench, cache_dir=str(DATA_DIR))
    print(f"Loaded {bench}: {len(datasets[bench])} samples")

## Summary Statistics

Overview of each benchmark: number of samples, average question length (in characters), and answer type.

In [ ]:
summary_rows = []
for bench in BENCHMARKS:
    data = datasets[bench]
    questions = [d["question"] if isinstance(d, dict) else d.question for d in data]
    lengths = [len(q) for q in questions]
    summary_rows.append({
        "Benchmark": bench,
        "Num Samples": len(data),
        "Mean Question Length (chars)": f"{np.mean(lengths):.0f}",
        "Median Question Length (chars)": f"{np.median(lengths):.0f}",
        "Min Length": min(lengths),
        "Max Length": max(lengths),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.set_index("Benchmark", inplace=True)
summary_df

## Question Length Distributions

Histograms showing the distribution of question lengths (in characters) for each benchmark.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Question Length Distributions by Benchmark", fontsize=16, fontweight="bold")

for ax, bench in zip(axes.flat, BENCHMARKS):
    data = datasets[bench]
    questions = [d["question"] if isinstance(d, dict) else d.question for d in data]
    lengths = [len(q) for q in questions]
    
    ax.hist(lengths, bins=20, color=sns.color_palette()[0], edgecolor="white", alpha=0.8)
    ax.set_title(bench.replace("_", " ").title(), fontsize=13)
    ax.set_xlabel("Question Length (chars)")
    ax.set_ylabel("Count")
    ax.axvline(np.mean(lengths), color="red", linestyle="--", label=f"Mean={np.mean(lengths):.0f}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../results/figures/data_exploration_lengths.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Example Questions

Display a few example questions from each benchmark to understand the problem types.

In [ ]:
NUM_EXAMPLES = 3

for bench in BENCHMARKS:
    data = datasets[bench]
    print(f"\n{'='*80}")
    print(f"  {bench.replace('_', ' ').upper()}")
    print(f"{'='*80}")
    for i, sample in enumerate(data[:NUM_EXAMPLES]):
        q = sample["question"] if isinstance(sample, dict) else sample.question
        # Truncate long questions for display
        display_q = q[:500] + "..." if len(q) > 500 else q
        print(f"\n--- Example {i+1} ---")
        print(display_q)
        
        # Show answer if available
        if isinstance(sample, dict) and "answer" in sample:
            ans = str(sample["answer"])
            print(f"  Answer: {ans[:100]}")
        elif hasattr(sample, "answer"):
            print(f"  Answer: {str(sample.answer)[:100]}")

## Word-Level Length Distribution

Additional view: question lengths measured in words rather than characters.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

all_lengths = {}
for bench in BENCHMARKS:
    data = datasets[bench]
    questions = [d["question"] if isinstance(d, dict) else d.question for d in data]
    all_lengths[bench] = [len(q.split()) for q in questions]

positions = np.arange(len(BENCHMARKS))
bp = ax.boxplot(
    [all_lengths[b] for b in BENCHMARKS],
    positions=positions,
    widths=0.5,
    patch_artist=True,
)

colors = sns.color_palette("Set2", len(BENCHMARKS))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

ax.set_xticks(positions)
ax.set_xticklabels([b.replace("_", " ").title() for b in BENCHMARKS])
ax.set_ylabel("Question Length (words)")
ax.set_title("Question Length Distribution (Words) by Benchmark", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()